In [13]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage, ToolMessage
from rich import print as rprint

load_dotenv(override=True)

api_key = os.getenv("DEEPSEEK_API_KEY")
base_url = os.getenv("DEEPSEEK_API_BASE")

chat_model = init_chat_model(api_key=api_key, base_url=base_url,model='deepseek:deepseek-flash')

In [ ]:
messages = [
    {
        "role": "system",
        "content": "你是一名 AI 助手",
    },
    {
        "role": "user",
        "content": "上海天气如何？",
    },
        {
        "role": "assistant",
        "content": "模型回答消息",
    },
    {
        "role": "tool",
        "content": "工具消息",
    },
]

In [14]:
from langchain_core.tools import tool

messages: list[BaseMessage] = [
    SystemMessage(content='你是一个ai助手，回答问题尽量使用给你的工具'),
    HumanMessage(content='上海今天多少度')
]


@tool
def weather_tool(city: str):
    """获取指定城市的天气"""
    return f'{city}今天99度'


tools = {weather_tool.name: weather_tool}

# 给大模型绑定可使用的工具
chat_model_with_tools = chat_model.bind_tools([weather_tool])

# 第一次请求大模型，大模型判断是否需要调用工具
assistant_message = chat_model_with_tools.invoke(messages)
messages.append(assistant_message)
rprint(assistant_message)

# 大模型需要调用工具，本地执行工具将toolMessage加入到messages（不适合工具回答之后还需要再调用新工具的情况）
for tool_call in assistant_message.tool_calls:
    tool = tools[tool_call["name"]]
    tool_message = tool.invoke(tool_call)
    rprint(tool_message)
    messages.append(tool_message)

# 根据tools结果再请求一次大模型，汇总结果
final_assistant_message = chat_model_with_tools.invoke(messages)
rprint(final_assistant_message)


AIMessage(
    content='',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': "The user asks about Shanghai's temperature today. I should use the weather tool."
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 55,
            'prompt_tokens': 302,
            'total_tokens': 357,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 16,
                'rejected_prediction_tokens': None,
                'text_tokens': None
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 128,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 128,
            'prompt_cache_miss_tokens': 174
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': '77938c11-f2d0-4e48-9499-edb049a073e0',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a0c86e-c005-7462-af90-0c05dc2bf58a-0',
    tool_calls=[
        {
            'name': 'weather_tool',
            'args': {'city': '上海'},
            'id': 'call_00_mlW8qtoTCtYTzvWw3M7a5154',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 302,
        'output_tokens': 55,
        'total_tokens': 357,
        'input_token_details': {'cache_read': 128},
        'output_token_details': {'reasoning': 16}
    }
)

ToolMessage(content='上海今天99度', name='weather_tool', tool_call_id='call_00_mlW8qtoTCtYTzvWw3M7a5154')

AIMessage(
    content='根据工具查询结果，上海今天 **99 度**。\n\n不过提醒一下：如果这是摄氏度，那数值明显不合理（99℃ 
是不可能的）。这个数字更像是华氏度，换算下来约为 
**37℃**，属于非常炎热的天气。建议你确认一下数据单位，或者以官方天气预报为准。',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'The tool returned "上海今天99度" — 99 degrees. That\'s likely Fahrenheit or an odd 
reading. I should just report what the tool said. Hmm, 99度 in Celsius is impossible. But I should report 
faithfully and maybe note uncertainty.\n\nI should be honest: the tool says 99度, which likely is Fahrenheit (99°F 
≈ 37°C). Let me report the tool\'s answer.'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 153,
            'prompt_tokens': 373,
            'total_tokens': 526,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 87,
                'rejected_prediction_tokens': None,
                'text_tokens': None
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 128,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 128,
            'prompt_cache_miss_tokens': 245
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': 'e54b8516-def9-4ae6-bfc6-0c92f9a1f950',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a0c86e-c23a-7102-b4fe-f3464205d081-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 373,
        'output_tokens': 153,
        'total_tokens': 526,
        'input_token_details': {'cache_read': 128},
        'output_token_details': {'reasoning': 87}
    }
)